In [ ]:
!pip install --quiet pytorch-lightning optuna optuna-integration[pytorch-lightning] mlflow torchinfo plotly boto3

In [ ]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms, models
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
import optuna
from optuna.integration import PyTorchLightningPruningCallback
from optuna.samplers import TPESampler
from optuna.pruners import HyperbandPruner
from optuna.trial import TrialState
import optuna.visualization as vis
import numpy as np
from PIL import Image
import os
import mlflow
from pytorch_lightning.loggers import MLFlowLogger
from torchinfo import summary
from contextlib import redirect_stdout
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
import warnings
from sklearn.metrics import precision_score, recall_score, fbeta_score, matthews_corrcoef, confusion_matrix, classification_report
import matplotlib.pyplot as plt
# import seaborn as sns
import json
import boto3

pl.seed_everything(42)
warnings.filterwarnings('ignore')

In [3]:
# Google Colab Settings
# from google.colab import drive
# drive.mount('/content/gdrive')
# TB_CLASSIFIER_MAIN_DIR = Path('/content/gdrive/MyDrive/TBClassifier')
# GDRIVE_DATASET_PATH = TB_CLASSIFIER_MAIN_DIR/"dataset/images"
# MLFLOW_TRACKING_DIR =TB_CLASSIFIER_MAIN_DIR/"mlflow"
# TRACKING_DB = MLFLOW_TRACKING_DIR/'tracking.db'
# OPTUNA_DIR = TB_CLASSIFIER_MAIN_DIR/"optuna_studies"
# STUDY_DB = OPTUNA_DIR / "optuna_study.db"

# TB_CLASSIFIER_MAIN_DIR.mkdir(parents=True, exist_ok=True)
# MLFLOW_TRACKING_DIR.mkdir(parents=True, exist_ok=True)
# OPTUNA_DIR.mkdir(parents=True, exist_ok=True)

# mlflow.set_tracking_uri(f"sqlite:///{TRACKING_DB}")
# mlflow.set_experiment("TBClassifier_tuning")

In [5]:
# local Settings
TB_CLASSIFIER_MAIN_DIR = Path('C:/Users/zheng/Desktop/TBClassifier')
GDRIVE_DATASET_PATH = TB_CLASSIFIER_MAIN_DIR/"dataset/images"
MLFLOW_TRACKING_DIR =TB_CLASSIFIER_MAIN_DIR/"mlflow"
TRACKING_DB = MLFLOW_TRACKING_DIR/'tracking.db'
OPTUNA_DIR = TB_CLASSIFIER_MAIN_DIR/"optuna_studies"
STUDY_DB = OPTUNA_DIR / "optuna_study.db"

TB_CLASSIFIER_MAIN_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_TRACKING_DIR.mkdir(parents=True, exist_ok=True)
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{TRACKING_DB}")
mlflow.set_experiment("TBClassifier_tuning")

<Experiment: artifact_location='file:C:/Users/zheng/Desktop/TBClassifier/mlruns/1', creation_time=1773710751256, experiment_id='1', last_update_time=1773710751256, lifecycle_stage='active', name='TBClassifier_tuning', tags={}, workspace='default'>

In [5]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
experiments = client.search_experiments()
for exp in experiments:
    print(f"Name: {exp.name}, ID: {exp.experiment_id}")

Name: TBClassifier_tuning, ID: 1
Name: Default, ID: 0


In [6]:
class DataPipeline(pl.LightningDataModule):
    def __init__(self, dataset_dir, batch_size=16, img_size=512, fold_idx=0, n_splits=5):
        super().__init__()
        self.dataset_dir = dataset_dir
        self.batch_size = batch_size
        self.img_size = img_size
        self.fold_idx = fold_idx
        self.n_splits = n_splits

        self.train_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((img_size, img_size)),
            transforms.RandomRotation(10),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        self.val_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def setup(self, stage=None):
        # Load all paths and labels
        train_dir = self.dataset_dir / 'train'
        all_paths = []
        all_labels = []

        folder_map = {'Healthy': 0, 'SickNonTB': 1, 'TB': 2}
        for folder, label in folder_map.items():
            folder_path = train_dir / folder
            if folder_path.exists():
                paths = list(folder_path.rglob('*.png'))
                all_paths.extend(paths)
                all_labels.extend([label] * len(paths))
                print(f"Found {len(paths)} in {folder}")

        all_labels = np.array(all_labels)

        # Stratified split
        skf = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=42)
        splits = list(skf.split(np.arange(len(all_paths)), all_labels))
        train_idx, val_idx = splits[self.fold_idx]

        # Create datasets
        train_paths = [all_paths[i] for i in train_idx]
        val_paths = [all_paths[i] for i in val_idx]
        test_paths = list((self.dataset_dir / 'test').rglob('*.png'))

        # Use simple PathDataset - no Subset inheritance!
        self.train_dataset = PathDataset(train_paths, self.train_transform)
        self.val_dataset = PathDataset(val_paths, self.val_transform)
        self.test_dataset = PathDataset(test_paths, self.val_transform)

        print(f"\nDataPipeline(): Fold {self.fold_idx + 1}/{self.n_splits}:")
        print(f"DataPipeline(): Train: {len(train_paths)}")
        print(f"DataPipeline(): Val:   {len(val_paths)}")
        print(f"DataPipeline(): Test:  {len(test_paths)}")

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=0)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=0)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=0)


class PathDataset(Dataset):

    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        image = Image.open(img_path).convert('L')

        # Get label from parent folder
        folder = img_path.parent.name
        label = {'Healthy': 0, 'SickNonTB': 1, 'TB': 2}.get(folder, 2)

        # Apply transform
        if self.transform:
            image = self.transform(image)

        return image, label

In [7]:
# Test
dm = DataPipeline(
    dataset_dir=GDRIVE_DATASET_PATH,
    batch_size=4,
    fold_idx=0,
    n_splits=5
)
dm.setup()

# Verify single sample
x, y = dm.train_dataset[0]
print(f"Single sample: type={type(x)}, shape={x.shape if hasattr(x, 'shape') else 'N/A'}, dtype={x.dtype if hasattr(x, 'dtype') else 'N/A'}")
from torch.utils.data import DataLoader
loader = DataLoader(dm.train_dataset, batch_size=4, num_workers=0)
batch_x, batch_y = next(iter(loader))
print(f"Batch: shape={batch_x.shape}, dtype={batch_x.dtype}")
print("SUCCESS!")

Found 3420 in Healthy
Found 3420 in SickNonTB
Found 720 in TB

DataPipeline(): Fold 1/5:
DataPipeline(): Train: 6048
DataPipeline(): Val:   1512
DataPipeline(): Test:  840
Single sample: type=<class 'torch.Tensor'>, shape=torch.Size([3, 512, 512]), dtype=torch.float32
Batch: shape=torch.Size([4, 3, 512, 512]), dtype=torch.float32
SUCCESS!


In [ ]:
class DenseNetClassifier(pl.LightningModule):
    def __init__(self, learning_rate, dropout, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        self.backbone = models.densenet121(weights="IMAGENET1K_V1")

        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(self.hparams.dropout),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(self.hparams.dropout),
            nn.Linear(512, 3)
        )

        # Normalized Inverse Frequency for 3800 Healthy, 3800 SickNonTB, 800 TB
        class_weights = torch.tensor([1.00, 1.00, 4.75])
        self.register_buffer('class_weights', class_weights)

        self.val_preds = []
        self.val_targets = []

    def forward(self, x):
        return self.backbone(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        loss = F.cross_entropy(logits, y, weight=self.class_weights)

        self.log('train_acc', acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)

        # print(f"DenseNetClassifier(): \ntrain_acc:  {acc}")
        # print(f"DenseNetClassifier(): train_loss:  {loss}")

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        loss = F.cross_entropy(logits, y, weight=self.class_weights)

        tb_precision = self.compute_tb_precision(y, preds)
        tb_recall = self.compute_tb_recall(y, preds)

        self.log('val_acc', acc, on_epoch=True, prog_bar=True)
        self.log('val_loss', loss, on_epoch=True,prog_bar=True)
        self.log('val_TB_precision', tb_precision, on_epoch=True, prog_bar=True)
        self.log('val_TB_recall', tb_recall, on_epoch=True, prog_bar=True)

        # print(f"DenseNetClassifier(): \nval_acc:  {acc}")
        # print(f"DenseNetClassifier(): val_loss:  {loss}")
        # print(f"DenseNetClassifier(): val_TB_precision:  {tb_precision}")
        # print(f"DenseNetClassifier(): val_TB_recall:  {tb_recall}")

        self.val_preds.append(preds.detach().cpu())
        self.val_targets.append(y.detach().cpu())

        return loss

    def on_validation_epoch_end(self):

        preds = torch.cat(self.val_preds)
        targets = torch.cat(self.val_targets)

        try:
            mcc = matthews_corrcoef(targets.numpy(), preds.numpy())
        except Exception:
            print("Error determining MCC")
            mcc = 0.0

        self.log("val_mcc", mcc, prog_bar=True)
        # print(f"DenseNetClassifier(): val_mcc:  {mcc}")

        self.val_preds.clear()
        self.val_targets.clear()

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        loss = F.cross_entropy(logits, y, weight=self.class_weights)

        tb_class = 2
        tb_precision = precision_score(y.cpu().numpy(), preds.cpu().numpy(), labels=[tb_class], average="macro", zero_division=0)
        tb_recall = recall_score(y.cpu().numpy(), preds.cpu().numpy(), labels=[tb_class], average="macro", zero_division=0)
        tb_f2 = fbeta_score(y.cpu().numpy(), preds.cpu().numpy(), beta=2, labels=[tb_class], average="macro", zero_division=0)
        try:
            mcc = matthews_corrcoef(y.cpu().numpy(), preds.cpu().numpy())
        except Exception:
            mcc = 0.0

        self.log("test_loss", loss)
        self.log("test_acc", acc)
        self.log("test_TB_precision", tb_precision)
        self.log("test_TB_recall", tb_recall)
        self.log("test_TB_f2", tb_f2)
        self.log("test_mcc", mcc)

        return {
            "test_loss": loss.detach(),
            "test_acc": acc.detach(),
            "test_TB_precision": torch.tensor(tb_precision, device=y.device),
            "test_TB_recall": torch.tensor(tb_recall, device=y.device),
            "test_TB_f2": torch.tensor(tb_f2, device=y.device),
            "test_mcc": torch.tensor(mcc, device=y.device)
        }

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.learning_rate,
            weight_decay=self.hparams.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
                'interval': 'epoch'
            }
        }

    # # Compute TB precision: TP / (TP + FP)
    # def compute_tb_precision(self, y_true, y_pred, tb_class=2):
    #   tp = ((y_true == tb_class) & (y_pred == tb_class)).sum()
    #   fp = ((y_true != tb_class) & (y_pred == tb_class)).sum()
    #   if (tp + fp) > 0:
    #       return tp.float() / (tp + fp).float()
    #   else:
    #       return torch.tensor(0.0, device=y_true.device)

    # # Compute TB recall: TP / (TP + FN)
    # def compute_tb_recall(self, y_true, y_pred, tb_class=2):
    #     tp = ((y_true == tb_class) & (y_pred == tb_class)).sum()
    #     fn = ((y_true == tb_class) & (y_pred != tb_class)).sum()
    #     if (tp + fn) > 0:
    #         return tp.float() / (tp + fn).float()
    #     else:
    #         return torch.tensor(0.0, device=y_true.device)

    # # Compute TB F2 score using sklearn (macro-average)
    # def compute_tb_f2(self, y_true, y_pred, tb_class=2):
    #     y_true_np = y_true.cpu().numpy()
    #     y_pred_np = y_pred.cpu().numpy()
    #     return torch.tensor(fbeta_score(y_true_np, y_pred_np, beta=2, labels=[tb_class], average='macro'), device=y_true.device)

In [9]:
def objective(trial):

    hyperparams = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True),
        'dropout': trial.suggest_float('dropout', 0.0, 0.5),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [16, 32])
    }

    with mlflow.start_run(nested=True) as run:

        mlflow.log_params({'trial_number': trial.number})
        mlflow.log_params(hyperparams)

        print(f"\n{'='*70}")
        print(f"Trial {trial.number}: {hyperparams}")
        print(f"MLflow Run ID: {run.info.run_id}")
        print(f"{'='*70}")

        fold_scores = {
            'val_acc': [],
            'val_loss': [],
            'val_TB_precision': [],
            'val_TB_recall': [],
            'val_mcc': [],
            'test_acc': [],
        }

        for fold_idx in range(5):
            print(f"\n--- Fold {fold_idx + 1}/5 ---")

            datamodule = DataPipeline(
                dataset_dir=GDRIVE_DATASET_PATH,
                batch_size=hyperparams['batch_size'],
                fold_idx=fold_idx,
                n_splits=5
            )

            model = DenseNetClassifier(
                learning_rate=hyperparams['learning_rate'],
                dropout=hyperparams['dropout'],
                weight_decay=hyperparams['weight_decay']
            )

            early_stop = EarlyStopping(
                monitor='val_mcc',
                mode='max',
                patience=5,
                verbose=False
            )

            pruning = PyTorchLightningPruningCallback(trial, monitor='val_mcc')

            trainer = pl.Trainer(
                max_epochs=30,
                accelerator='gpu' if torch.cuda.is_available() else 'cpu',
                devices=1,
                callbacks=[early_stop, pruning],
                enable_progress_bar=True,
                logger=MLFlowLogger(
                    experiment_name="TBClassifier_tuning",
                    tracking_uri=f"sqlite:///{TRACKING_DB}"
                ),
                deterministic=True
            )

            try:
                trainer.fit(model, datamodule=datamodule)
                val_metrics = trainer.validate(model, datamodule=datamodule, verbose=False)[0]

                fold_scores['val_acc'].append(val_metrics['val_acc'])
                fold_scores['val_loss'].append(val_metrics['val_loss'])
                fold_scores['val_TB_precision'].append(val_metrics['val_TB_precision'])
                fold_scores['val_TB_recall'].append(val_metrics['val_TB_recall'])
                fold_scores['val_mcc'].append(val_metrics['val_mcc'])

                mlflow.log_metrics({
                    f'fold_{fold_idx}_best_epoch': trainer.current_epoch,
                    f'fold_{fold_idx}_val_acc': val_metrics['val_acc'],
                    f'fold_{fold_idx}_val_loss': val_metrics['val_loss'],
                    f'fold_{fold_idx}_val_TB_precision': val_metrics['val_TB_precision'],
                    f'fold_{fold_idx}_val_TB_recall': val_metrics['val_TB_recall'],
                    f'fold_{fold_idx}_val_mcc': val_metrics['val_mcc']
                }, step=fold_idx)

                intermediate_value = np.mean(fold_scores['val_mcc'])
                trial.report(intermediate_value, step=fold_idx)

                if trial.should_prune():
                    print(f"Trial pruned at fold {fold_idx + 1}")
                    raise optuna.TrialPruned()

            except Exception as e:
                print(f"Fold {fold_idx + 1} failed: {e}")
                fold_scores['val_acc'].append(0.0)
                fold_scores['val_loss'].append(float('inf'))
                fold_scores['val_TB_precision'].append(0.0)
                fold_scores['val_TB_recall'].append(0.0)
                fold_scores['val_mcc'].append(0.0)

        mean_val_acc = np.mean(fold_scores['val_acc'])
        mean_val_loss = np.mean(fold_scores['val_loss'])
        mean_val_TB_precision = np.mean(fold_scores['val_TB_precision'])
        mean_val_TB_recall = np.mean(fold_scores['val_TB_recall'])
        mean_val_mcc = np.mean(fold_scores['val_mcc'])

        print(f"mean_val_acc: {mean_val_acc}")
        print(f"mean_val_loss: {mean_val_loss}")
        print(f"mean_val_TB_precision: {mean_val_TB_precision}")
        print(f"mean_val_TB_recall: {mean_val_TB_recall}")
        print(f"mean_val_mcc: {mean_val_mcc}")

        mlflow.log_metrics({
            'mean_val_acc': mean_val_acc,
            'mean_val_loss': mean_val_loss,
            'mean_val_TB_precision': mean_val_TB_precision,
            'mean_val_TB_recall': mean_val_TB_recall,
            'mean_val_mcc': mean_val_mcc
        })

        model_summary = str(model)
        mlflow.log_text(model_summary, f"model_summary_trial_{trial.number}.txt")

        return mean_val_mcc

In [10]:
def run_optimization(n_trials, timeout, n_splits, max_epochs_per_fold):

    print(f"\n{'='*70}")
    print("OPTUNA OPTIMIZATION SETUP")
    print(f"{'='*70}")
    print(f"run_optimization(): Trials: {n_trials}")
    print(f"run_optimization(): Timeout: {timeout}s ({timeout/3600:.1f} hours)")
    print(f"run_optimization(): CV Folds: {n_splits}")
    print(f"run_optimization(): Max epochs per fold: {max_epochs_per_fold}")
    print(f"run_optimization(): Sampler: TPE (n_startup={5}, n_candidates={24})")
    print(f"run_optimization(): Pruner: Hyperband (min_resource={3}, max_resource={max_epochs_per_fold}, reduction_factor={3})")

    study = optuna.create_study(
        study_name="TBClassifier",
        direction='maximize',
        storage=f"sqlite:///{STUDY_DB}",
        sampler=TPESampler(
            n_startup_trials=5,      # Random exploration first
            n_ei_candidates=24       # Bayesian optimization candidates
        ),
        pruner=HyperbandPruner(
            min_resource=3,            # At least 3 epochs before pruning
            max_resource=max_epochs_per_fold,  # Max epochs per trial
            reduction_factor=3         # Aggressive pruning factor
        ),
        load_if_exists=True
    )

     # Run optimization
    print(f"\n{'='*70}")
    print("STARTING OPTIMIZATION")
    print(f"{'='*70}")

    study.optimize(
        objective,
        n_trials=n_trials,
        timeout=timeout,
        show_progress_bar=True,
        catch=(Exception,)  # Continue if single trial fails
    )

    with mlflow.start_run(run_name="STUDY_SUMMARY"):
        mlflow.log_params({
            'n_trials_requested': n_trials,
            'timeout_seconds': timeout,
            'n_splits': n_splits,
            'max_epochs_per_fold': max_epochs_per_fold,
            'sampler': 'TPE',
            'sampler_n_startup': 5,
            'sampler_n_candidates': 24,
            'pruner': 'Hyperband',
            'pruner_min_resource': 3,
            'pruner_max_resource': max_epochs_per_fold,
            'pruner_reduction_factor': 3,
        })

        mlflow.log_metrics({
            'trials_completed': len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]),
            'trials_pruned': len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]),
            'trials_failed': len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL]),
            'best_trial_number': study.best_trial.number if study.best_trial else -1,
            'best_value': study.best_value if study.best_trial else 0.0
        })

        if study.best_trial:
            mlflow.log_dict({
                'best_params': study.best_params,
                'best_value': study.best_value,
                'best_trial_number': study.best_trial.number,
            }, artifact_file="best_params.json")

    print(f"\n{'='*70}")
    print("OPTIMIZATION COMPLETE")
    print(f"{'='*70}")

    if study.best_trial:
        print(f"Best Trial: #{study.best_trial.number}")
        print(f"Best Value: {study.best_value:.4f}")
        print("Best Hyperparameters:")
        for key, value in study.best_params.items():
            print(f"  {key}: {value}")
    else:
        print("No successful trials completed!")

    return study

In [11]:
# if os.path.exists(STUDY_DB):
#     print(f"Deleting existing Optuna study database: {STUDY_DB}")
#     os.remove(STUDY_DB)

study = run_optimization(
    n_trials=50,
    timeout=3600*5,
    n_splits=5,
    max_epochs_per_fold=30
)

print(f"Best trial: {study.best_trial.number}")
print(f"Best params: {study.best_params}")

[I 2026-03-17 09:26:04,712] A new study created in RDB with name: TBClassifier



OPTUNA OPTIMIZATION SETUP
run_optimization(): Trials: 50
run_optimization(): Timeout: 18000s (5.0 hours)
run_optimization(): CV Folds: 5
run_optimization(): Max epochs per fold: 30
run_optimization(): Sampler: TPE (n_startup=5, n_candidates=24)
run_optimization(): Pruner: Hyperband (min_resource=3, max_resource=30, reduction_factor=3)

STARTING OPTIMIZATION


  0%|          | 0/50 [00:00<?, ?it/s]


Trial 0: {'learning_rate': 9.074558463205873e-05, 'dropout': 0.12708700265655404, 'weight_decay': 0.00011819483743829251, 'batch_size': 16}
MLflow Run ID: 13c3870c0a824ed8ac735fd1369a87d6

--- Fold 1/5 ---


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 3090 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Found 3420 in Healthy
Found 3420 in SickNonTB
Found 720 in TB

DataPipeline(): Fold 5/5:
DataPipeline(): Train: 6048
DataPipeline(): Val:   1512
DataPipeline(): Test:  840


DenseNetClassifier(): val_mcc:  0.9931754455643025

mean_val_acc: 0.9924603343009949
mean_val_loss: 0.02690487541258335
mean_val_TB_precision: 0.10052909702062607
mean_val_TB_recall: 0.09854497313499451
mean_val_mcc: 0.9871161818504334
[I 2026-03-17 15:30:06,530] Trial 2 finished with value: 0.9871161818504334 and parameters: {'learning_rate': 2.1461011888319234e-05, 'dropout': 0.3244594323856504, 'weight_decay': 0.00011857385227303369, 'batch_size': 16}. Best is trial 2 with value: 0.9871161818504334.

OPTIMIZATION COMPLETE
Best Trial: #2
Best Value: 0.9871
Best Hyperparameters:
  learning_rate: 2.1461011888319234e-05
  dropout: 0.3244594323856504
  weight_decay: 0.00011857385227303369
  batch_size: 16
Best trial: 2
Best params: {'learning_rate': 2.1461011888319234e-05, 'dropout': 0.3244594323856504, 'weight_decay': 0.00011857385227303369, 'batch_size': 16}


In [13]:
# Query completed trials (same pattern as your previous project)
complete_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]
pruned_trials = [t for t in study.trials if t.state == TrialState.PRUNED]

print(f"Complete trials: {len(complete_trials)}")
print(f"Pruned trials: {len(pruned_trials)}")
print(f"Best trial: #{study.best_trial.number}")

# Detailed best trial info
best_trial = study.best_trial
print(f"  Value (active-TB accuracy): {best_trial.value:.4f}")
print("  Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

Complete trials: 3
Pruned trials: 0
Best trial: #2
  Value (active-TB accuracy): 0.9871
  Hyperparameters:
    learning_rate: 2.1461011888319234e-05
    dropout: 0.3244594323856504
    weight_decay: 0.00011857385227303369
    batch_size: 16


In [8]:
study = optuna.load_study(
    study_name="TBClassifier",
    storage=f"sqlite:///{STUDY_DB}"
)

In [15]:
print("Trials:", len(study.trials))
for t in study.trials:
    print(t.number, t.state)

Trials: 3
0 TrialState.COMPLETE
1 TrialState.COMPLETE
2 TrialState.COMPLETE


In [12]:
vis.plot_param_importances(study)

In [13]:
vis.plot_slice(study, params=["learning_rate", "dropout", "weight_decay", "batch_size"])

In [11]:
vis.plot_optimization_history(study)

In [19]:
study.best_params

{'learning_rate': 2.1461011888319234e-05,
 'dropout': 0.3244594323856504,
 'weight_decay': 0.00011857385227303369,
 'batch_size': 16}

In [17]:
with open("best_params.json", "w") as f:
    json.dump(study.best_params, f)

mlflow.log_artifact("best_params.json")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

# Read keys from environment
aws_access_key = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
region = os.getenv("AWS_REGION")

In [ ]:
s3 = boto3.client(
    "s3",
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    region_name=region
)
BUCKET = "tb-classifier-artifacts-506261418229-ap-southeast-2-an"
s3.upload_file("optuna_studies/optuna_study.db", BUCKET, "tuning/optuna_study.db")
s3.upload_file("mlflow.db", BUCKET, "tuning/mlflow.db")
s3.upload_file("best_params.json", BUCKET, "tuning/best_params.json")

In [ ]:
# best_params = study.best_params
# best_model= DenseNetClassifier(
#     learning_rate=best_params['learning_rate'],
#     dropout=best_params['dropout'],
#     weight_decay=best_params['weight_decay']
# )
# dm = DataPipeline(batch_size=best_params['batch_size'])

# # Full training with best config
# trainer = pl.Trainer(
#     max_epochs=50,
#     accelerator='gpu',
#     devices=1,
#     logger=MLFlowLogger(
#         experiment_name="TBClassifier_Best",
#         tracking_uri="file:///content/mlruns",
#         run_name=f"best-trial-{study.best_trial.number}"
#     ),
#     callbacks=[
#         EarlyStopping(monitor='val_loss', patience=10, mode='min'),
#         ModelCheckpoint(
#             monitor='val_mcc',
#             filename='best_tb_model',
#             save_top_k=1,
#             mode='max'
#         )
#     ]
# )

# trainer.fit(final_model, dm)
# trainer.test(final_model, dm)

# mlflow.pytorch.log_model(final_model, "final_model")

# # Export to ONNX for Triton serving later
# final_model.to_onnx(
#     file_path='tb_densenet121.onnx',
#     input_sample=torch.randn(1, 3, 512, 512),
#     export_params=True,
#     opset_version=17
# )

In [ ]:
# Log to MLflow model registry
# mlflow.set_tracking_uri("file:///content/mlruns")
# with mlflow.start_run(run_name="production_candidate"):
#     mlflow.log_params(best_params)
#     mlflow.log_metrics({
#         'final_val_acc': trainer.callback_metrics['val_acc'].item(),
#         'final_val_acc_active_tb': trainer.callback_metrics['val_acc_active-tb'].item()
#     })
#     mlflow.pytorch.log_model(final_model, "model")
#     mlflow.onnx.log_model(onnx_model="tb_densenet121.onnx", artifact_path="onnx_model")

In [ ]:
# final_model = DenseNetClassifier(
#     learning_rate=best_trial.params['learning_rate'],
#     dropout=best_trial.params['dropout'],
#     freeze_backbone=False
# )

# os.makedirs("model_stats", exist_ok=True)

# model_stats = summary(
#     final_model,
#     input_size=(1, 3, 512, 512),
#     verbose=0
# )

# print("\nModel Summary:")
# print(f"Total params: {model_stats.total_params:,}")
# print(f"Trainable params: {model_stats.trainable_params:,}")

# # Save to file
# with open(f"model_stats/best_trial_{best_trial.number}_summary.txt", 'w') as f:
#     with redirect_stdout(f):
#         print(f"Best Trial: #{best_trial.number}")
#         print(f"Active-TB Accuracy: {best_trial.value:.4f}")
#         print("\nHyperparameters:")
#         for key, value in best_trial.params.items():
#             print(f"  {key}: {value}")
#         print(f"\nTotal params: {model_stats.total_params:,}")
#         print(f"Trainable params: {model_stats.trainable_params:,}")
#         print(model_stats)

In [ ]:
# Register Model in MLFLow
# Train Best Model
# Plot Loss curve
# Confusion Matrix
# F2 Beta
# Grad CAM